# RecruitBrain — Colab Precompute (RAM-safe chunked version)
**Steps:**
1. Cell 1 — install deps
2. Cell 2 — upload 4 split files
3. Cell 3 — combine + verify 100K lines
4. Cell 4 — embed in chunks (RAM-safe, ~15 min)
5. Cell 5 — build FAISS + verify + download

**Output files → drop into `artifacts/` on your machine:**
- `candidate_embeddings.npy`
- `faiss.index`
- `candidate_ids_colab.json`

In [ ]:
# Cell 1: Install
!pip install fastembed faiss-cpu numpy -q
print('Install done')

In [ ]:
# Cell 2: Upload 4 split files
# First run this on your LOCAL machine:
#   cd "/home/kartik/Kartik/Claude-projects/H2S-Hackthon/[PUB] India_runs_data_and_ai_challenge/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge"
#   split -l 25000 candidates.jsonl candidates_part_
# Upload all 4 parts: candidates_part_aa, candidates_part_ab, candidates_part_ac, candidates_part_ad
from google.colab import files
print('Upload the 4 split files now...')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# Cell 3: Combine parts + verify
import os

parts = sorted([f for f in os.listdir('.') if f.startswith('candidates_part_')])
print('Parts found:', parts)

with open('candidates.jsonl', 'w', encoding='utf-8') as out:
    for part in parts:
        with open(part, 'r', encoding='utf-8') as f:
            out.write(f.read())

with open('candidates.jsonl', encoding='utf-8') as f:
    total = sum(1 for line in f if line.strip())

print(f'Total lines: {total}')
assert total == 100000, f'Expected 100000, got {total}'
print('OK — all 100K candidates present')

In [ ]:
# Cell 4: RAM-safe chunked embedding
# Processes 5000 candidates at a time, saves chunk embeddings to disk
# Peak RAM: ~2GB instead of 12GB

import json, gc, os
import numpy as np
from fastembed import TextEmbedding

CHUNK_SIZE = 5000
OUTPUT_DIR = 'chunks'
os.makedirs(OUTPUT_DIR, exist_ok=True)

def build_text(c):
    prof = c['profile']
    parts = []
    parts.append(f"Title: {prof.get('current_title','')}. {prof.get('headline','')}. "
                 f"Company: {prof.get('current_company','')}. YOE: {prof.get('years_of_experience',0)}.")
    summary = prof.get('summary', '')
    if summary:
        parts.append(summary[:500])
    for role in c.get('career_history', [])[:3]:
        desc = role.get('description', '')
        if desc:
            parts.append(f"Role at {role.get('company','')} as {role.get('title','')}: {desc[:300]}")
    skill_parts = [f"{s.get('name','')} ({s.get('proficiency','')})" for s in c.get('skills', [])[:15]]
    if skill_parts:
        parts.append('Skills: ' + ', '.join(skill_parts))
    return ' '.join(p for p in parts if p)

print('Loading model...')
model = TextEmbedding('BAAI/bge-small-en-v1.5')

all_ids = []
chunk_num = 0
chunk_texts = []
chunk_ids = []

with open('candidates.jsonl', encoding='utf-8') as f:
    for i, line in enumerate(f):
        line = line.strip()
        if not line:
            continue
        try:
            c = json.loads(line)
        except json.JSONDecodeError:
            continue

        chunk_texts.append(build_text(c))
        chunk_ids.append(c['candidate_id'])

        if len(chunk_texts) == CHUNK_SIZE:
            emb = np.array(list(model.embed(chunk_texts, batch_size=256)), dtype=np.float32)
            # L2 normalize
            norms = np.linalg.norm(emb, axis=1, keepdims=True)
            emb = emb / np.where(norms == 0, 1.0, norms)
            # Save chunk
            np.save(f'{OUTPUT_DIR}/chunk_{chunk_num:03d}.npy', emb)
            all_ids.extend(chunk_ids)
            print(f'  Chunk {chunk_num} done ({len(all_ids)} total, shape {emb.shape})')
            chunk_num += 1
            chunk_texts = []
            chunk_ids = []
            del emb
            gc.collect()

# Process remaining
if chunk_texts:
    emb = np.array(list(model.embed(chunk_texts, batch_size=256)), dtype=np.float32)
    norms = np.linalg.norm(emb, axis=1, keepdims=True)
    emb = emb / np.where(norms == 0, 1.0, norms)
    np.save(f'{OUTPUT_DIR}/chunk_{chunk_num:03d}.npy', emb)
    all_ids.extend(chunk_ids)
    print(f'  Chunk {chunk_num} done ({len(all_ids)} total)')
    del emb
    gc.collect()

# Save IDs
with open('candidate_ids_colab.json', 'w') as f:
    json.dump(all_ids, f)

print(f'\nAll chunks done. Total candidates: {len(all_ids)}')
print(f'Chunk files: {sorted(os.listdir(OUTPUT_DIR))}')

In [ ]:
# Cell 5: Merge chunks + build FAISS + verify + download
import numpy as np
import faiss
import json
import os
import gc

OUTPUT_DIR = 'chunks'

# Merge all chunk .npy files
print('Merging chunks...')
chunk_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.npy')])
print(f'Found {len(chunk_files)} chunks')

arrays = [np.load(f'{OUTPUT_DIR}/{f}') for f in chunk_files]
embeddings = np.concatenate(arrays, axis=0)
del arrays
gc.collect()

print(f'Merged embeddings shape: {embeddings.shape}')
np.save('candidate_embeddings.npy', embeddings)
print('Saved: candidate_embeddings.npy')

# Build FAISS
print('Building FAISS index...')
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
faiss.write_index(index, 'faiss.index')
print(f'Saved: faiss.index ({index.ntotal} vectors)')

# Verify
ids = json.load(open('candidate_ids_colab.json'))
print(f'\nEmbeddings : {embeddings.shape}')
print(f'FAISS      : {index.ntotal}')
print(f'IDs        : {len(ids)}')
assert embeddings.shape[0] == index.ntotal == len(ids), 'MISMATCH'
assert embeddings.shape[1] == 384
print('\nALL OK — downloading 3 files...')

from google.colab import files
files.download('candidate_embeddings.npy')
files.download('faiss.index')
files.download('candidate_ids_colab.json')